# Figure 5a — compounds detected in 2D vs 3D

Stacked bars of how many compounds pass the grit threshold in 2D only, in 3D
(single-cell aggregates) only, or in both, for each cell line.

**Provenance.** Upstream this panel lived in `3_Figure3/GritScores/3_GritScores.ipynb`
(cell 18), which read four `grit_scores_descriptive_stats_{data_type}_{cell_line}.csv`
files. The cell that *wrote* those CSVs does not survive in any source tree, and
`3_GritScores` cannot run here anyway (`cytominer-eval==0.1` imports `np.float`,
removed in numpy 1.24).

The compound sets are recoverable without it: they are exactly the treatments whose
**median grit per perturbation** exceeds 1.96, read from the published
`grit_data_{data_type}_{cell_line}.parquet` tables. That definition was validated
against all four surviving upstream CSVs and reproduces every set exactly
(HCT116 2D 46/46, HT29 2D 47/47, HCT116 aggregates 38/38, HT29 aggregates 33/33).


In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import profiles
from utils.panels import save_panel

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
GRIT_THRESHOLD = 1.96
CELL_LINES = ['HCT116', 'HT29']
TOTAL_COMPOUNDS = 52


def passing_compounds(data_type, cell_line, threshold=GRIT_THRESHOLD):
    """Treatments whose median grit per perturbation clears the threshold.

    Median per (compound, concentration) — not per replicate row — which is what
    reproduces the upstream descriptive-stats CSVs exactly.
    """
    df = pd.read_parquet(profiles("exp1_main", f"grit_data_{data_type}_{cell_line}.parquet"))
    trt = df[df["Metadata_pert_type"] == "trt"]
    per_pert = trt.groupby(["Metadata_name", "Metadata_pert_name"])["Metadata_grit"].median()
    return set(per_pert[per_pert > threshold].reset_index()["Metadata_name"])


sets = {(dt, cl): passing_compounds(dt, cl)
        for dt in ("2D", "aggregates") for cl in CELL_LINES}
for (dt, cl), s in sets.items():
    print(f"{dt:11s} {cl:7s} n={len(s)}")

In [ ]:
# Common / unique-to-2D / unique-to-3D per cell line
rows = []
for cl in CELL_LINES:
    s2d, s3d = sets[("2D", cl)], sets[("aggregates", cl)]
    rows.append({"cell_line": cl,
                 "common":    len(s2d & s3d),
                 "unique_2D": len(s2d - s3d),
                 "unique_3D": len(s3d - s2d)})
counts = pd.DataFrame(rows).set_index("cell_line")
counts

In [ ]:
fig = plt.figure(figsize=(2, 3))
sns.set_style("whitegrid")

colors = sns.color_palette("gray", 4)[1:]   # skip the lightest

x = counts.index.tolist()
common, u2d, u3d = counts["common"], counts["unique_2D"], counts["unique_3D"]

plt.bar(x, common, color=colors[2])
plt.bar(x, u2d, bottom=common, color=colors[1])
plt.bar(x, u3d, bottom=np.array(common) + np.array(u2d), color=colors[0])

plt.axhline(y=TOTAL_COMPOUNDS, color='k', linestyle='--', linewidth=1)
plt.ylabel("# compounds")
plt.legend([f'total compounds={TOTAL_COMPOUNDS}', 'Common', 'Unique to 2D', 'Unique to 3D'],
           bbox_to_anchor=(1.01, 1), loc='upper left')

save_panel(fig, "Fig5a", data=counts.reset_index(),
           caption="Compounds passing grit: common, unique to 2D, unique to 3D",
           notebook="analysis/3_Figure5/3_Fig5a_grit_overlap.ipynb")
plt.show()